# **HW 2. Nonlinear models: feature importance from a tree to boostings**

### What is inside

This notebook is the practice for the lessons **"Random Forest vs Decision Tree importances"**, **"Practical subtleties of Feature importances: Catboost"** and **"Practical subtleties of Feature importances: XGB and LightGBM"**. We will go through the whole chain of questions about feature importance reconstructed from the structure of a model:

1. what exactly averaging gives us when we go from a tree to a forest — and why it is about variance rather than bias;
2. how impurity importance systematically lies in favour of "rich" features, and how to see that with your own hands;
3. how many trees are actually needed before the importances stop wobbling;
4. why one trained XGBoost model gives three different answers to "which feature is important";
5. the four CatBoost importances and the single one that can be negative;
6. what to do with all of this in practice.

The formula the first block is built around is the variance of the mean of $M$ trees with pairwise correlation $\rho$:

$$\operatorname{Var}\!\left(\frac{1}{M}\sum_{m=1}^{M} a_m\right) = \rho\,\sigma^2 + \frac{1-\rho}{M}\,\sigma^2.$$

### **A reminder: what we can already do**

By this point we already know how to collect importance from the structure of a single tree:

___
1. **Impurity importance** — the sum of the reductions of the criterion (Gini, entropy, MSE) over all the splits where the feature takes part, weighted by the number of objects in the node.
2. **The importance of a forest** — the same quantity averaged over $M$ trees. This is exactly what sits in sklearn's `feature_importances_`.
3. **The importance of a boosting** — by construction no longer about "purity", but about the reduction of the loss function itself.
___

And we know two limitations that are not going anywhere:

- impurity importance is computed **on the training sample** — it is about how much the feature helped to fit, not about how useful it is on new data;
- it is **biased** towards features with a large number of possible split thresholds.

We will see both of them in numbers in this notebook.

To start with, let us gather everything we will need.

In [ ]:
!pip install -q catboost


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import spearmanr

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool

Let us fix the randomness.

In [ ]:
RANDOM_STATE = 42

As the dataset we take [Pima Indians Diabetes](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database) — the same one the importance tables in the theory are built on. The target variable `Outcome` is whether the patient has diabetes.

A detailed EDA is not needed here: the data is already numeric and has no missing values in the usual sense.

In [ ]:
path = 'https://github.com/SadSabrina/open-xai-materials/raw/refs/heads/main/data/diabetes.csv'
data = pd.read_csv(path)

X = data.drop(columns='Outcome')
y = data['Outcome']

X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=RANDOM_STATE, test_size=0.25, stratify=y
)

labels = list(X_train.columns)
print('train:', X_train.shape, ' test:', X_test.shape)

## Block 1. Tree → forest: what exactly averaging gives us

In the theory we decomposed the error into three parts and showed that averaging $M$ trees hits one single term — the variance. Let us check that by hand.

First, the basic comparison: a single tree against a forest.

In [ ]:
tree = DecisionTreeClassifier(random_state=RANDOM_STATE)
tree.fit(X_train, y_train)

forest = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
forest.fit(X_train, y_train)

print('ROC-AUC of a single tree:', round(roc_auc_score(y_test, tree.predict_proba(X_test)[:, 1]), 3))
print('ROC-AUC of the forest   :', round(roc_auc_score(y_test, forest.predict_proba(X_test)[:, 1]), 3))

Now the main part. The variance in the decomposition is the spread **from one training sample to another**. So to see it, we have to retrain the model many times on different samples and look at the spread of the importances.

Let us make 30 bootstrap repeats and train both a tree and a forest on each of them.

In [ ]:
def importances_over_resamples(make_model, n_repeats=30, seed=RANDOM_STATE):
    """Feature importances collected over n_repeats bootstrap samples."""
    rng = np.random.RandomState(seed)
    rows = []
    for r in range(n_repeats):
        idx = rng.choice(len(X_train), size=len(X_train), replace=True)
        model = make_model(r)
        model.fit(X_train.iloc[idx], y_train.iloc[idx])
        rows.append(model.feature_importances_)
    return pd.DataFrame(rows, columns=labels)

imp_tree = importances_over_resamples(
    lambda r: DecisionTreeClassifier(random_state=RANDOM_STATE)
)
imp_forest = importances_over_resamples(
    lambda r: RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
)

spread = pd.DataFrame({
    'mean_tree': imp_tree.mean(),
    'std_tree': imp_tree.std(ddof=1),
    'mean_forest': imp_forest.mean(),
    'std_forest': imp_forest.std(ddof=1),
})
spread['std_ratio'] = spread['std_tree'] / spread['std_forest']
spread.sort_values('mean_forest', ascending=False).round(4)

**Q1.** Take the feature that is most important according to the forest (`mean_forest`). How many times larger is its importance spread `std_tree` than `std_forest`? Round the answer to one decimal place.



In [ ]:
# your code here


Look at the top rows of the table: for the strong features `mean_tree` and `mean_forest` are close, while `std_forest` is 2–3 times smaller than `std_tree`. This is exactly what the theory promised: **averaging damps the variance, not the bias**. The ordering of the features and the means themselves hold, the wobble around them drops.

(For the weak features the means diverge more noticeably — but that is no longer about averaging, it is about random subsets of features giving them a chance to land in a split that a single greedy tree never got round to.)

**Q2.** What changed when we went from a single tree to a forest?

`Select all correct statements in the trainer`

Now let us check the formula itself. The variance of the mean:

$$\operatorname{Var}\!\left(\frac{1}{M}\sum_m a_m\right) = \rho\,\sigma^2 + \frac{1-\rho}{M}\,\sigma^2,$$

where $\sigma^2$ is the variance of a single tree's prediction and $\rho$ is the pairwise correlation of the trees. Let us extract both straight from the trained forest: every tree inside `forest.estimators_` has its own prediction.

In [ ]:
def tree_predictions(fit_forest, X):
    """Matrix (number of trees x number of objects) with each tree's predictions."""
    Xv = X.values if hasattr(X, 'values') else X   # the trees inside the forest were fitted on an array
    return np.vstack([t.predict_proba(Xv)[:, 1] for t in fit_forest.estimators_])

def sigma2_and_rho(fit_forest, X):
    P = tree_predictions(fit_forest, X)
    sigma2 = P.var(axis=0, ddof=1).mean()          # spread across trees, averaged over objects
    corr = np.corrcoef(P)                          # pairwise correlations of the trees
    off = corr[np.triu_indices_from(corr, k=1)]
    return sigma2, np.nanmean(off)

sigma2, rho = sigma2_and_rho(forest, X_test)
print('sigma^2 =', round(sigma2, 4))
print('rho     =', round(rho, 4))

**Q3.** What is the pairwise correlation of the trees $\rho$ in the forest `forest`? Round the answer to two decimal places.

In [ ]:
# your code here


Let us substitute into the formula and see how many times averaging knocked the variance down — and where it goes in the limit.

In [ ]:
M = forest.n_estimators

var_mean = rho * sigma2 + (1 - rho) / M * sigma2
var_limit = rho * sigma2                     # the limit as M -> infinity

print(f'variance of a single tree  : {sigma2:.4f}')
print(f'variance of the mean (M={M}): {var_mean:.4f}   -> {sigma2/var_mean:.1f}x smaller')
print(f'limit as M -> inf          : {var_limit:.4f}   -> {1/rho:.1f}x smaller, nowhere left to fall')

This is where the whole construction of a random forest grows from. The second term $\frac{1-\rho}{M}\sigma^2$ is damped by the number of trees — that is free, just add trees. But the first one, $\rho\sigma^2$, cannot be removed by the number of trees at all: however many trees you add, the variance will not fall below it.

The only way to break through this floor is to reduce $\rho$ itself. This is exactly why a forest picks the split at every node not over all the features but over a random subset. Let us check that this works: we will train a forest that is allowed to look at all the features at once.

In [ ]:
forest_all = RandomForestClassifier(
    n_estimators=100, max_features=None, random_state=RANDOM_STATE
)
forest_all.fit(X_train, y_train)

sigma2_all, rho_all = sigma2_and_rho(forest_all, X_test)

pd.DataFrame({
    'max_features': ['sqrt (default)', 'None (all features)'],
    'rho': [round(rho, 3), round(rho_all, 3)],
    'variance floor rho*sigma^2': [round(rho * sigma2, 4), round(rho_all * sigma2_all, 4)],
})

**Q4.** What is $\rho$ for the forest with `max_features=None`? Round the answer to two decimal places.

Compare it with Q3 and explain the sign of the difference to yourself: trees that search over the same features at every node arrive at similar splits — and therefore at similar predictions.

In [ ]:
# your code here


## Block 2. The bias of impurity importance

In the theory we went through three sources of bias. The most noticeable one is the **skew towards "rich" features**: the more possible split thresholds a feature has, the higher the chance that some threshold will by accident give a large reduction of impurity.

Let us check that in its pure form. We will add to the data three features that are **in no way related to the target variable** and differ only in the number of possible splits:

- `random_num` — continuous, with almost as many thresholds as there are objects;
- `random_cat_50` — categorical with 50 levels;
- `random_cat_2` — binary, with one single possible threshold.

If the importance were honest, all three would get roughly zero.

In [ ]:
rng = np.random.RandomState(RANDOM_STATE)

def add_dummies(df, rng):
    out = df.copy()
    out['random_num'] = rng.normal(size=len(df))
    out['random_cat_50'] = rng.randint(0, 50, size=len(df))
    out['random_cat_2'] = rng.randint(0, 2, size=len(df))
    return out

X_train_d = add_dummies(X_train, rng)
X_test_d = add_dummies(X_test, rng)

labels_d = list(X_train_d.columns)
X_train_d.head()

In [ ]:
forest_d = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
forest_d.fit(X_train_d, y_train)

imp_d = pd.Series(forest_d.feature_importances_, index=labels_d).sort_values(ascending=False)

plt.figure(figsize=(9, 4))
colors = ['#eb6834' if f.startswith('random_') else '#2a78d6' for f in imp_d.index]
bar = plt.barh(imp_d.index[::-1], imp_d.values[::-1], color=colors[::-1])
plt.title('Impurity importance: orange marks the deliberately meaningless features', pad=12)
plt.tight_layout();

In [ ]:
imp_d.round(4).to_frame('impurity_importance')

**Q5.** Which of the three dummy features received the highest impurity importance?

`Select one option in the trainer`

**Q6.** What place does `random_num` take in the overall importance ranking (1 being the most important)? The answer is an integer.

In [ ]:
# your code here


The difference between the three dummy features is not an accident and not luck. It is exactly what the theory predicts: the more candidate splits a feature has, the higher the maximum Gini gain **even with no signal at all**. This is the effect of a maximally selected statistic, the same in nature as multiple testing: test enough thresholds and one of them will fire by chance.

Let us count the candidates explicitly. An important caveat: sklearn cannot do real categorical splits — it sees the integer-coded `random_cat_50` as an ordinary number and cuts it with a threshold. So for all three features the number of candidates is simply the number of distinct values minus one.

**Q7.** Why did `random_num` beat `random_cat_2`, even though both are equally meaningless?

`Select all correct statements in the trainer`

In [ ]:
n_thresholds = pd.DataFrame({
    'distinct values': [X_train_d[c].nunique() for c in ['random_num', 'random_cat_50', 'random_cat_2']],
    'impurity importance': [round(imp_d[c], 4) for c in ['random_num', 'random_cat_50', 'random_cat_2']],
}, index=['random_num', 'random_cat_50', 'random_cat_2'])
n_thresholds['candidate splits'] = n_thresholds['distinct values'] - 1
n_thresholds[['distinct values', 'candidate splits', 'impurity importance']]

Now the second source of bias — **estimation on the training sample**. Impurity importance is assembled from how much the feature helped to fit `X_train`. Let us ask a different question: how much will the model degrade on the **test** set if we destroy this feature?

This is permutation importance — the first post-hoc method we will meet. Here we need it as a control.

In [ ]:
perm = permutation_importance(
    forest_d, X_test_d, y_test,
    n_repeats=30, random_state=RANDOM_STATE, scoring='roc_auc'
)

compare = pd.DataFrame({
    'impurity (on train)': forest_d.feature_importances_,
    'permutation (on test)': perm.importances_mean,
}, index=labels_d).sort_values('impurity (on train)', ascending=False)

compare.round(4)

**Q8.** By impurity importance `random_num` took 5th place (Q6). What place does it take by permutation importance on the test set? The answer is an integer.

Note the sign as well: for a meaningless feature the permutation importance goes negative — by destroying it the model gets slightly **better** on the test set.

In [ ]:
# your code here


## Block 3. How many trees are enough

We know that the spread of the importances falls as $\frac{1-\rho}{M}$. So there is a point beyond which adding trees is almost pointless: the second term is already small compared with the floor $\rho\sigma^2$.

Let us look at this directly — at how the stability of the importances depends on the number of trees.

In [ ]:
def importance_std_for_M(M, n_repeats=15, seed=RANDOM_STATE):
    rng = np.random.RandomState(seed)
    rows = []
    for r in range(n_repeats):
        idx = rng.choice(len(X_train), size=len(X_train), replace=True)
        f = RandomForestClassifier(n_estimators=M, random_state=RANDOM_STATE, n_jobs=-1)
        f.fit(X_train.iloc[idx], y_train.iloc[idx])
        rows.append(f.feature_importances_)
    return pd.DataFrame(rows, columns=labels).std(ddof=1).mean()

M_grid = [1, 2, 5, 10, 25, 50, 100, 200, 400]
stds = [importance_std_for_M(M) for M in M_grid]

stability = pd.DataFrame({'M': M_grid, 'mean std of importances': np.round(stds, 4)})
stability

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(M_grid, stds, marker='o', color='#2a78d6')
plt.xscale('log')
plt.xlabel('number of trees M (log scale)')
plt.ylabel('mean spread of importances')
plt.title('The spread of importances plateaus — more trees barely help', pad=12)
plt.grid(alpha=0.3)
plt.tight_layout();

**Q9.** How many times smaller is the mean spread of the importances at $M=100$ than at $M=1$? Round the answer to one decimal place.

In [ ]:
# your code here


And now look at the right edge of the table. Between $M=100$ and $M=400$ the spread barely changes — the curve has plateaued. This is $\rho\sigma^2$: the term $\frac{1-\rho}{M}\sigma^2$ is already so small that new trees give nothing.

The practical meaning is exactly the one built into the `async_mode` of the `rfgboost` package: stop adding trees once the fall has stopped, and get the same accuracy with fewer trees.

## Block 4. Boostings: one model, three different answers

We move on to boostings. Here the importance is computed not by "purity" any more, but by the reduction of the loss function — and there are several ways to compute it at once. XGBoost has three:

- **gain** — how much the splits on the feature improved the objective function;
- **cover** — how many objects (more precisely, what total Hessian) those splits touch;
- **weight** — how many times the feature was used at all.

Let us train one model and extract all three from it.

In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.1,
    random_state=RANDOM_STATE, eval_metric='logloss'
)
xgb_model.fit(X_train, y_train)

booster = xgb_model.get_booster()

xgb_imp = pd.DataFrame({
    t: pd.Series(booster.get_score(importance_type=t))
    for t in ['gain', 'total_gain', 'cover', 'total_cover', 'weight']
}).reindex(labels)

xgb_imp.round(2)

It is already visible that the columns rank the features differently. Let us measure that honestly — with the Spearman rank correlation.

In [ ]:
for a, b in [('gain', 'weight'), ('gain', 'cover'), ('cover', 'weight')]:
    r = spearmanr(xgb_imp[a], xgb_imp[b]).statistic
    print(f'Spearman({a:6s}, {b:6s}) = {r: .2f}')

**Q10.** What is the Spearman rank correlation between the rankings by `gain` and by `weight`? Round the answer to two decimal places.

In [ ]:
# your code here


**Q11.** Find the feature that rises noticeably higher by `weight` than by `gain` (compare the places in the two rankings). What does that mean?

`Select all correct statements in the trainer`

In [ ]:
ranks = pd.DataFrame({
    'place by gain': xgb_imp['gain'].rank(ascending=False).astype(int),
    'place by weight': xgb_imp['weight'].rank(ascending=False).astype(int),
})
ranks['shift'] = ranks['place by gain'] - ranks['place by weight']
ranks.sort_values('shift', ascending=False)

Let us check one more statement from the theory: `cover` is `total_cover` divided by the number of splits on the feature, that is by `weight`.

**Q12.** Take the feature `Glucose`. What is `total_cover / weight`? Round the answer to the nearest integer and compare it with the `cover` column.

In [ ]:
# your code here


Now LightGBM. In the theory we said that its `split` is exactly XGBoost's `weight` in meaning (the number of splits), while `gain` differs in the way it is aggregated. The meaning is the same — the numbers are different.

In [ ]:
lgb_model = lgb.LGBMClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.1,
    random_state=RANDOM_STATE, verbose=-1
)
lgb_model.fit(X_train, y_train)

lgb_imp = pd.DataFrame({
    'split': lgb_model.booster_.feature_importance(importance_type='split'),
    'gain': lgb_model.booster_.feature_importance(importance_type='gain'),
}, index=labels)

both = pd.DataFrame({
    'XGB weight': xgb_imp['weight'],
    'LGBM split': lgb_imp['split'],
    'XGB total_gain': xgb_imp['total_gain'].round(1),
    'LGBM gain': lgb_imp['gain'].round(1),
})
both

**Q13.** Do the **numbers** in the `XGB weight` and `LGBM split` columns coincide? And the feature rankings by them?

`Select the correct statement in the trainer`

In [ ]:
print('numbers coincide    :', bool((both['XGB weight'] == both['LGBM split']).all()))
print('Spearman over ranks :', round(spearmanr(both['XGB weight'], both['LGBM split']).statistic, 2))

## Block 5. CatBoost: four importances and one negative

CatBoost has four ways of computing importance. We are interested in the two that show up on tabular data:

- **PredictionValuesChange** — how much the prediction changes on average when the feature changes. It is computed by default and normalized so that the sum over all features equals 100.
- **LossFunctionChange** — how much the **metric** will change if the feature is removed from the model. Unlike the first one, this importance knows in which direction the prediction moved — and therefore **can be negative**.

In [ ]:
cb_model = CatBoostClassifier(
    iterations=200, depth=4, learning_rate=0.1,
    random_seed=RANDOM_STATE, verbose=0
)
cb_model.fit(X_train, y_train)

train_pool = Pool(X_train, y_train)

cb_imp = pd.DataFrame({
    'PredictionValuesChange': cb_model.get_feature_importance(type='PredictionValuesChange'),
    'LossFunctionChange (train)': cb_model.get_feature_importance(train_pool, type='LossFunctionChange'),
}, index=labels).sort_values('PredictionValuesChange', ascending=False)

cb_imp.round(4)

**Q14.** What is the sum of `PredictionValuesChange` over all the features? The answer is an integer.

In [ ]:
# your code here


Note: on the training Pool there are no negative values at all. That is understandable — on the very data the model was trained on, every feature helped the fit at least a little. This is exactly that second source of bias the lesson spoke about.

For the importance to start answering the question "is this feature needed on new data at all?", it has to be computed on a **held-out** sample. Let us take the data with the dummy features from Block 2 for that — then it is immediately visible whom the metric ought to reject.

In [ ]:
cb_model_d = CatBoostClassifier(
    iterations=200, depth=4, learning_rate=0.1,
    random_seed=RANDOM_STATE, verbose=0
)
cb_model_d.fit(X_train_d, y_train)

lfc = pd.DataFrame({
    'on train': cb_model_d.get_feature_importance(Pool(X_train_d, y_train), type='LossFunctionChange'),
    'on test': cb_model_d.get_feature_importance(Pool(X_test_d, y_test), type='LossFunctionChange'),
}, index=labels_d).sort_values('on test')

print('negative on train:', int((lfc['on train'] < 0).sum()))
print('negative on test  :', int((lfc['on test'] < 0).sum()))
lfc.round(4)

**Q15.** Which feature turned out to have the most negative `LossFunctionChange` on the test Pool?

`Select one option in the trainer`

A negative value reads literally: removing the feature **improved** the metric. None of the importances we computed before can say that in principle — they all measure "how much the feature took part", not "did it make things better". And `random_num`, which by impurity importance was cheerfully sitting in 5th place, finally gets what it deserves here.

In [ ]:
# your code here


And the last thing worth holding on to about CatBoost: the fourth importance, **PredictionDiff**, is arranged in a fundamentally different way — it is computed **for a pair of objects**, only for continuous features, and explains not the model as a whole but a specific difference between two observations. By type it is a **local** importance, unlike every other one in this notebook.

**Q16.** Match each method with its type.

`Do the matching in the trainer`

## Block 6. What to do with all of this

We have computed feature importance in eight different ways on one and the same data. Let us collect the top-3 of three model families and see how much they agree with each other at all.

In [ ]:
top3 = {
    'RandomForest': list(pd.Series(forest.feature_importances_, index=labels).nlargest(3).index),
    'XGBoost (gain)': list(xgb_imp['gain'].nlargest(3).index),
    'CatBoost (PVC)': list(cb_imp['PredictionValuesChange'].nlargest(3).index),
    'LightGBM (gain)': list(lgb_imp['gain'].nlargest(3).index),
}

for name, feats in top3.items():
    print(f'{name:17s}: {feats}')

common = set(top3['RandomForest']) & set(top3['XGBoost (gain)']) & set(top3['CatBoost (PVC)'])
print()
print('top-3 intersection across the three families:', sorted(common))

**Q17.** How many features made it into the top-3 of RandomForest, XGBoost and CatBoost at once? The answer is an integer.

In [ ]:
# your code here


**Q18.** What is the most correct way to estimate feature importance?

`Select all correct statements in the trainer`

### Conclusions

Let us go through what we saw with our own eyes:

1. **Averaging works against variance, not against bias.** The spread of the importances of a forest fell several times over compared with a single tree, while the mean values stayed roughly the same. The formula $\rho\sigma^2 + \frac{1-\rho}{M}\sigma^2$ also shows the limit: the variance will never fall below $\rho\sigma^2$, however many trees you add.

2. **Random subsets of features are not decoration but a way to break through that limit.** A forest with `max_features=None` gives noticeably more correlated trees, and therefore a higher variance floor.

3. **Impurity importance systematically lies in favour of "rich" features.** Three deliberately meaningless features received radically different importances, and the order between them is set by the number of possible splits, not by any relation to the target variable. Permutation importance on the test set puts them back in their place.

4. **One trained model has several different answers to "which feature is important".** gain, cover and weight are three different questions, not three ways of answering one. Between frameworks even importances with the same name do not coincide.

5. **A negative importance is not a bug.** LossFunctionChange can say "it would have been better without this feature", because it is the only one tied to the metric rather than to the fact of taking part in splits.

The overall conclusion is the same one the whole block has been leading us to: the more flexible the model, the more ways there are to "reconstruct" importance from how it is built — and the less agreement there is among them. This is exactly the motivation for post-hoc methods, which the next part of the course starts with: they ask the question from outside the model and therefore do not depend on how it is arranged inside.

That is all, friends! Well done.

See you in the next homeworks, \
Your course team : )